In [ ]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_19'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

# 4. Lesson 4 Hands-On Lab — Containerization Fundamentals

Uses `boto3` only — no SageMaker SDK.

**Build strategy (3-layer fallback):**

| Layer | Method | Condition |
|---|---|---|
| 1 | **AWS CodeBuild** | Always tried first — no local Docker needed |
| 2 | **Local Docker** (`subprocess`) | If CodeBuild unavailable/fails |
| 3 | **Print commands** | Always shown — run in any terminal |

**AWS services:** S3 · ECR · CodeBuild · ECS · CloudWatch Logs · CloudWatch Metrics


## 4.1 Environment Setup

### Block 1 — Import Libraries

In [1]:
# Block 1 - Import libraries

from google.colab import userdata

def get_colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f'Unable to read Colab Secret: {name}') from exc
        return None
    if required and not value:
        raise RuntimeError(f'Add the Colab Secret {name} and grant this notebook access.')
    return value

import io
import json
import base64
import zipfile
import time
import datetime as dt
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

print("Libraries imported")

Libraries imported


### Block 2 — Connect to AWS

In [2]:
# Fetch AWS credentials from Colab secrets
AWS_ACCESS_KEY_ID = get_colab_secret('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = get_colab_secret('AWS_SECRET_ACCESS_KEY')
AWS_SESSION_TOKEN = get_colab_secret('AWS_SESSION_TOKEN', required=False)

os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ['AWS_SESSION_TOKEN'] = AWS_SESSION_TOKEN

print("AWS credentials loaded from Colab secrets and set as environment variables.")

# Block 2 - Create AWS clients

session = boto3.Session()
region  = session.region_name or "us-east-2"

s3           = session.client("s3",         region_name=region)
ecr          = session.client("ecr",        region_name=region)
codebuild    = session.client("codebuild",  region_name=region)
ecs          = session.client("ecs",        region_name=region)
logs_client  = session.client("logs",       region_name=region)
cloudwatch   = session.client("cloudwatch", region_name=region)
sts          = session.client("sts",        region_name=region)
iam          = session.client("iam",        region_name=region)

identity   = sts.get_caller_identity()
account_id = identity["Account"]
caller_arn = identity["Arn"]

# Get role name from caller ARN
if ":assumed-role/" in caller_arn:
    role_name = caller_arn.split(":assumed-role/")[1].split("/")[0]
else:
    role_name = caller_arn.split("/")[-1]

# Use iam.get_role() to get the EXACT ARN including /service-role/ path
# Never construct this manually — the path varies per account
execution_role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]

print("Region         :", region)
print("Account        :", account_id)
print("Caller ARN     :", caller_arn)
print("Execution role :", execution_role_arn)

Region         : eu-north-1
Account        : 797715838180
Caller ARN     : arn:aws:sts::797715838180:assumed-role/AmazonSageMakerAdminIAMExecutionRole/SageMaker
Execution role : arn:aws:iam::797715838180:role/service-role/AmazonSageMakerAdminIAMExecutionRole


### Block 3 — Configure Project Paths

In [3]:
# Block 3 - Configure project names and S3 paths

project_name  = "lesson4-containerization"
run_id        = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
ecr_repo_name = "lesson4-loan-risk-model"
cb_project    = f"lesson4-docker-build-{account_id}"
namespace     = "Lesson4/Containerization"

bucket_name = f"{project_name}-{account_id}-{region}".replace("_", "-").lower()
prefix      = "lesson4/container-artifacts"

# S3 keys
dockerfile_key   = f"{prefix}/dockerfiles/Dockerfile.train"
multistage_key   = f"{prefix}/dockerfiles/Dockerfile.serve"
gpu_key          = f"{prefix}/dockerfiles/Dockerfile.gpu"
compose_key      = f"{prefix}/dockerfiles/docker-compose.yml"
requirements_key = f"{prefix}/dockerfiles/requirements.txt"
dockerignore_key = f"{prefix}/dockerfiles/.dockerignore"
source_zip_key   = f"{prefix}/build-source/source.zip"
manifest_key     = f"{prefix}/manifest/image_manifest_{run_id}.json"

local_dir = Path("lesson4_outputs")
local_dir.mkdir(exist_ok=True)

print("Bucket     :", bucket_name)
print("ECR repo   :", ecr_repo_name)
print("CB project :", cb_project)
print("Run ID     :", run_id)

Bucket     : lesson4-containerization-797715838180-eu-north-1
ECR repo   : lesson4-loan-risk-model
CB project : lesson4-docker-build-797715838180
Run ID     : 20260713-050858


## 4.2 S3 Bucket

### Block 4 — Create or Reuse S3 Bucket

In [4]:
# Block 4 - Create or reuse S3 bucket

def bucket_exists(name):
    try:
        s3.head_bucket(Bucket=name)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ["404","NoSuchBucket"]:
            return False
        raise

if not bucket_exists(bucket_name):
    if region == "us-east-2":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region}
        )
    print("Created bucket:", bucket_name)
else:
    print("Using existing bucket:", bucket_name)

s3.put_bucket_versioning(
    Bucket=bucket_name, VersioningConfiguration={"Status":"Enabled"}
)
print("S3 versioning enabled")

Using existing bucket: lesson4-containerization-797715838180-eu-north-1
S3 versioning enabled


## 4.3 Docker Artefacts

### Block 5 — requirements.txt and .dockerignore

In [5]:
# Block 5 - Write requirements.txt and .dockerignore

requirements_content = (
    "scikit-learn==1.4.2\n"
    "numpy==1.26.4\n"
    "pandas==2.2.2\n"
    "joblib==1.4.2\n"
    "fastapi==0.111.0\n"
    "uvicorn[standard]==0.29.0\n"
    "boto3==1.34.99\n"
    "prometheus-client==0.20.0\n"
)
dockerignore_content = (
    "__pycache__/\n*.py[cod]\n.git/\n.env\n"
    "tests/\nnotebooks/\n*.ipynb\ndata/\nmodels/\n*.joblib\n"
)

(local_dir / "requirements.txt").write_text(requirements_content)
(local_dir / ".dockerignore").write_text(dockerignore_content)
print("requirements.txt:")
print(requirements_content)

requirements.txt:
scikit-learn==1.4.2
numpy==1.26.4
pandas==2.2.2
joblib==1.4.2
fastapi==0.111.0
uvicorn[standard]==0.29.0
boto3==1.34.99
prometheus-client==0.20.0



### Block 6 — Training Dockerfile

In [6]:
# Block 6 - AI/ML Training Dockerfile (only copies files that exist in ZIP)

train_lines = [
    "FROM python:3.10-slim",
    "LABEL maintainer='mlops-team@company.com' version='1.0.0'",
    "",
    "ENV PYTHONDONTWRITEBYTECODE=1 \\",
    "    PYTHONUNBUFFERED=1 \\",
    "    PIP_NO_CACHE_DIR=1",
    "",
    "WORKDIR /app",
    "",
    "RUN apt-get update && apt-get install -y --no-install-recommends \\",
    "        gcc g++ curl \\",
    "    && rm -rf /var/lib/apt/lists/*",
    "",
    "COPY requirements.txt .",
    "RUN pip install -r requirements.txt",
    "",
    "COPY train.py .",
    "",
    "RUN useradd -m -u 1000 -s /bin/bash mluser \\",
    "    && chown -R mluser:mluser /app",
    "USER mluser",
    "",
    "EXPOSE 8080",
    "",
    'CMD ["python", "train.py"]',
]
dockerfile_train = "\n".join(train_lines)
(local_dir / "Dockerfile.train").write_text(dockerfile_train)
print("Dockerfile.train written")
print(dockerfile_train)

Dockerfile.train written
FROM python:3.10-slim
LABEL maintainer='mlops-team@company.com' version='1.0.0'

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
        gcc g++ curl \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY train.py .

RUN useradd -m -u 1000 -s /bin/bash mluser \
    && chown -R mluser:mluser /app
USER mluser

EXPOSE 8080

CMD ["python", "train.py"]


### Block 7 — Multi-Stage Serving Dockerfile

In [7]:
# Block 7 - Multi-stage serving Dockerfile (fixed)

serve_lines = [
    "FROM python:3.10 AS builder",
    "WORKDIR /build",
    "COPY requirements.txt .",
    "RUN pip install --no-cache-dir --target=/build/packages -r requirements.txt",
    "",
    "FROM python:3.10-slim AS runtime",
    "COPY --from=builder /build/packages /usr/local/lib/python3.10/site-packages/",
    "WORKDIR /app",
    "COPY requirements.txt .",
    "",
    "ENV PYTHONDONTWRITEBYTECODE=1 \\",
    "    PYTHONUNBUFFERED=1 \\",
    "    PORT=8080",
    "",
    "RUN useradd -m -u 1000 -s /bin/bash mluser \\",
    "    && chown -R mluser:mluser /app",
    "USER mluser",
    "",
    "EXPOSE 8080",
    "",
    'CMD ["python", "-m", "http.server", "8080"]',
]
dockerfile_serve = "\n".join(serve_lines)
(local_dir / "Dockerfile.serve").write_text(dockerfile_serve)
print("Dockerfile.serve written")
print(dockerfile_serve)

Dockerfile.serve written
FROM python:3.10 AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/build/packages -r requirements.txt

FROM python:3.10-slim AS runtime
COPY --from=builder /build/packages /usr/local/lib/python3.10/site-packages/
WORKDIR /app
COPY requirements.txt .

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PORT=8080

RUN useradd -m -u 1000 -s /bin/bash mluser \
    && chown -R mluser:mluser /app
USER mluser

EXPOSE 8080

CMD ["python", "-m", "http.server", "8080"]


### Block 8 — GPU Training Dockerfile

In [8]:
# Block 8 - GPU-enabled training Dockerfile

gpu_lines = [
    "FROM nvidia/cuda:11.8.0-cudnn8-runtime-ubuntu22.04",
    "ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 \\\\",
    "    CUDA_HOME=/usr/local/cuda \\\\",
    "    LD_LIBRARY_PATH=/usr/local/cuda/lib64 \\\\",
    "    NVIDIA_VISIBLE_DEVICES=all \\\\",
    "    NVIDIA_DRIVER_CAPABILITIES=compute,utility",
    "RUN apt-get update && apt-get install -y --no-install-recommends \\\\",
    "        python3.10 python3-pip curl \\\\",
    "    && rm -rf /var/lib/apt/lists/* \\\\",
    "    && ln -sf python3.10 /usr/bin/python3",
    "WORKDIR /app",
    "COPY requirements-gpu.txt .",
    "RUN pip install --no-cache-dir -r requirements-gpu.txt",
    "COPY . .",
    "RUN useradd -m -u 1000 mluser && chown -R mluser:mluser /app",
    "USER mluser",
    'CMD ["python", "train_gpu.py"]',
]
dockerfile_gpu = "\n".join(gpu_lines)
(local_dir / "Dockerfile.gpu").write_text(dockerfile_gpu)
gpu_req = "torch==2.2.0+cu118\ntorchvision==0.17.0+cu118\n--extra-index-url https://download.pytorch.org/whl/cu118\nscikit-learn==1.4.2\nnumpy==1.26.4\nboto3==1.34.99\n"
(local_dir / "requirements-gpu.txt").write_text(gpu_req)
print("Dockerfile.gpu written")

Dockerfile.gpu written


### Block 9 — docker-compose.yml

In [9]:
# Block 9 - docker-compose.yml (3-service AI stack)

compose_lines = [
    'version: "3.9"',
    "services:",
    "  trainer:",
    "    build: {dockerfile: Dockerfile.train}",
    "    volumes: [./data:/app/data:ro, model-store:/app/models]",
    "    networks: [ml-network]",
    '    restart: "no"',
    "  server:",
    "    build: {dockerfile: Dockerfile.serve, target: runtime}",
    '    ports: ["8080:8080"]',
    "    volumes: [model-store:/app/models:ro]",
    "    depends_on: [redis]",
    "    healthcheck:",
    '      test: ["CMD","curl","-f","http://localhost:8080/health"]',
    "      interval: 30s",
    "      retries: 3",
    "    networks: [ml-network]",
    "    restart: unless-stopped",
    "  redis:",
    "    image: redis:7-alpine",
    "    networks: [ml-network]",
    "networks:",
    "  ml-network: {driver: bridge}",
    "volumes:",
    "  model-store:",
]
compose_yml = "\n".join(compose_lines)
(local_dir / "docker-compose.yml").write_text(compose_yml)
print("docker-compose.yml written")
print(compose_yml)

docker-compose.yml written
version: "3.9"
services:
  trainer:
    build: {dockerfile: Dockerfile.train}
    volumes: [./data:/app/data:ro, model-store:/app/models]
    networks: [ml-network]
    restart: "no"
  server:
    build: {dockerfile: Dockerfile.serve, target: runtime}
    ports: ["8080:8080"]
    volumes: [model-store:/app/models:ro]
    depends_on: [redis]
    healthcheck:
      test: ["CMD","curl","-f","http://localhost:8080/health"]
      interval: 30s
      retries: 3
    networks: [ml-network]
    restart: unless-stopped
  redis:
    image: redis:7-alpine
    networks: [ml-network]
networks:
  ml-network: {driver: bridge}
volumes:
  model-store:


## 4.4 Upload to S3 + Build Source ZIP

### Block 10 — Upload Dockerfiles and Create Build Source ZIP

We package all files into a **ZIP archive** — this is the source CodeBuild will download, exactly how production CI/CD pipelines work with CodeBuild S3 sources.

In [10]:
# Block 10 - Upload artifacts + source ZIP (with train.py placeholder added)

artifacts = {
    dockerfile_key:   local_dir / "Dockerfile.train",
    multistage_key:   local_dir / "Dockerfile.serve",
    gpu_key:          local_dir / "Dockerfile.gpu",
    compose_key:      local_dir / "docker-compose.yml",
    requirements_key: local_dir / "requirements.txt",
    dockerignore_key: local_dir / ".dockerignore",
}
print("Uploading artefacts...")
for s3_key, local_path in artifacts.items():
    s3.upload_file(str(local_path), bucket_name, s3_key)
    print(f"  {local_path.name:<25} -> {s3_key}")

# train.py placeholder — real content would be the actual training script
train_py = (
    "# FinSight AI — Loan Risk Training Script\n"
    "# This placeholder shows the container builds and runs correctly.\n"
    "# In production, replace with the actual training code.\n"
    "import sys\n"
    "print('Container started successfully')\n"
    "print('Python:', sys.version)\n"
    "print('Training script would run here')\n"
)

buildspec_yaml = (
    "version: 0.2\n"
    "phases:\n"
    "  pre_build:\n"
    "    commands:\n"
    "      - echo Logging in to ECR\n"
    "      - aws ecr get-login-password --region $AWS_DEFAULT_REGION | docker login --username AWS --password-stdin $ECR_REPO_URI\n"
    "  build:\n"
    "    commands:\n"
    "      - docker build -t $ECR_REPO_URI:train-$RUN_ID -f Dockerfile.train .\n"
    "      - docker build -t $ECR_REPO_URI:serve-$RUN_ID --target runtime -f Dockerfile.serve .\n"
    "  post_build:\n"
    "    commands:\n"
    "      - docker push $ECR_REPO_URI:train-$RUN_ID\n"
    "      - docker push $ECR_REPO_URI:serve-$RUN_ID\n"
    "      - echo Build complete\n"
)

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("buildspec.yml",    buildspec_yaml)
    zf.writestr("Dockerfile.train", (local_dir / "Dockerfile.train").read_text())
    zf.writestr("Dockerfile.serve", (local_dir / "Dockerfile.serve").read_text())
    zf.writestr("requirements.txt", (local_dir / "requirements.txt").read_text())
    zf.writestr(".dockerignore",    (local_dir / ".dockerignore").read_text())
    zf.writestr("train.py",         train_py)    # ← added

zip_buffer.seek(0)
s3.upload_fileobj(zip_buffer, bucket_name, source_zip_key)
print(f"\nBuild source ZIP -> s3://{bucket_name}/{source_zip_key}")
print("  Contains: buildspec.yml, Dockerfile.train, Dockerfile.serve,")
print("            requirements.txt, .dockerignore, train.py")

Uploading artefacts...
  Dockerfile.train          -> lesson4/container-artifacts/dockerfiles/Dockerfile.train
  Dockerfile.serve          -> lesson4/container-artifacts/dockerfiles/Dockerfile.serve
  Dockerfile.gpu            -> lesson4/container-artifacts/dockerfiles/Dockerfile.gpu
  docker-compose.yml        -> lesson4/container-artifacts/dockerfiles/docker-compose.yml
  requirements.txt          -> lesson4/container-artifacts/dockerfiles/requirements.txt
  .dockerignore             -> lesson4/container-artifacts/dockerfiles/.dockerignore

Build source ZIP -> s3://lesson4-containerization-797715838180-eu-north-1/lesson4/container-artifacts/build-source/source.zip
  Contains: buildspec.yml, Dockerfile.train, Dockerfile.serve,
            requirements.txt, .dockerignore, train.py


## 4.5 Amazon ECR

### Block 11 — Create ECR Repository

In [11]:
# Block 11 - Create ECR repository

try:
    resp = ecr.create_repository(
        repositoryName=ecr_repo_name,
        tags=[
            {"Key": "Project",   "Value": project_name},
            {"Key": "RunId",     "Value": run_id},
            {"Key": "ManagedBy", "Value": "BITS-Pilani-Lab"},
        ],
        imageScanningConfiguration={"scanOnPush": True},
        encryptionConfiguration={"encryptionType": "AES256"}
    )
    repo = resp["repository"]
    print("Created ECR repository")

except ClientError as e:
    code = e.response["Error"]["Code"]

    if code == "RepositoryAlreadyExistsException":
        repo = ecr.describe_repositories(
            repositoryNames=[ecr_repo_name]
        )["repositories"][0]
        print("Reusing existing ECR repository")

    elif code == "AccessDeniedException":
        # ecr:CreateRepository not allowed — check if the repo already exists
        print("Warning: ecr:CreateRepository denied — checking for existing repository...")
        try:
            repo = ecr.describe_repositories(
                repositoryNames=[ecr_repo_name]
            )["repositories"][0]
            print(f"Found existing repository: {ecr_repo_name}")

        except ClientError as desc_err:
            if desc_err.response["Error"]["Code"] in (
                "RepositoryNotFoundException", "AccessDeniedException"
            ):
                # Neither create nor describe is permitted
                # Set placeholder URIs so downstream blocks don't NameError
                print()
                print("ECR access not available on this role.")
                print("Ask your AWS admin to run this once:")
                print(f"  aws ecr create-repository \\")
                print(f"    --repository-name {ecr_repo_name} \\")
                print(f"    --region {region} \\")
                print(f"    --image-scanning-configuration scanOnPush=true \\")
                print(f"    --encryption-configuration encryptionType=AES256")
                print()
                ecr_repo_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo_name}"
                ecr_repo_arn = f"arn:aws:ecr:{region}:{account_id}:repository/{ecr_repo_name}"
                registry_url = f"{account_id}.dkr.ecr.{region}.amazonaws.com"
                print(f"Continuing with placeholder URI:")
                print(f"  {ecr_repo_uri}")
                print("Re-run this block after the repo is created.")
                repo = None
            else:
                raise
    else:
        raise

# Set repo-level variables only when we have a real repository object
if repo is not None:
    ecr_repo_uri = repo["repositoryUri"]
    ecr_repo_arn = repo["repositoryArn"]
    registry_url = f"{account_id}.dkr.ecr.{region}.amazonaws.com"

    print()
    print("Repository Name :", repo["repositoryName"])
    print("Repository URI  :", ecr_repo_uri)
    print("Scan on push    :", repo["imageScanningConfiguration"]["scanOnPush"])
    print("Encryption      :", repo["encryptionConfiguration"]["encryptionType"])
    print()
    print("ECR Console:")
    print(f"https://{region}.console.aws.amazon.com/ecr/repositories/{ecr_repo_name}")

Reusing existing ECR repository

Repository Name : lesson4-loan-risk-model
Repository URI  : 797715838180.dkr.ecr.eu-north-1.amazonaws.com/lesson4-loan-risk-model
Scan on push    : True
Encryption      : AES256

ECR Console:
https://eu-north-1.console.aws.amazon.com/ecr/repositories/lesson4-loan-risk-model


### Block 12 — Configure ECR Scanning and Lifecycle Policy

In [12]:
# Block 12 - ECR scanning + lifecycle policy

try:
    ecr.put_image_scanning_configuration(
        repositoryName=ecr_repo_name,
        imageScanningConfiguration={"scanOnPush": True}
    )
    print("Image scanning enabled")
except Exception as e:
    if "AccessDenied" in str(e) or "AuthorizationError" in str(e):
        print(f"Scanning skipped (permission): {type(e).__name__}")
    else: raise

lifecycle_policy = {
    "rules": [
        {
            "rulePriority": 1,
            "description":  "Keep 10 most recent tagged images",
            "selection": {
                "tagStatus":     "tagged",
                "tagPrefixList": ["train-", "serve-", "v"],
                "countType":     "imageCountMoreThan",
                "countNumber":   10
            },
            "action": {"type": "expire"}
        },
        {
            "rulePriority": 2,
            "description":  "Delete untagged images after 7 days",
            "selection": {
                "tagStatus": "untagged",
                "countType": "sinceImagePushed",
                "countUnit": "days",
                "countNumber": 7
            },
            "action": {"type": "expire"}
        }
    ]
}
try:
    ecr.put_lifecycle_policy(
        repositoryName=ecr_repo_name,
        lifecyclePolicyText=json.dumps(lifecycle_policy)
    )
    print("Lifecycle policy applied (10 tagged / 7-day untagged)")
except Exception as e:
    if "AccessDenied" in str(e) or "AuthorizationError" in str(e):
        print(f"Lifecycle policy skipped (permission): {type(e).__name__}")
    else: raise

Image scanning enabled
Lifecycle policy applied (10 tagged / 7-day untagged)


### Block 13 — Retrieve ECR Authentication Token

In [13]:
# Block 13 - ECR auth token (valid 12 hours)

auth_resp      = ecr.get_authorization_token()
auth_data      = auth_resp["authorizationData"][0]
auth_token     = auth_data["authorizationToken"]
proxy_endpoint = auth_data["proxyEndpoint"]
token_expiry   = auth_data["expiresAt"]

decoded        = base64.b64decode(auth_token).decode("utf-8")
username, password = decoded.split(":", 1)

print("ECR Authentication Token Retrieved")
print(f"Registry  : {proxy_endpoint}")
print(f"Username  : {username}")
print(f"Password  : {password[:12]}... [truncated]")
print(f"Expires   : {token_expiry}")
print()
print("docker login command:")
print(f"  aws ecr get-login-password --region {region} | \\")
print(f"  docker login --username AWS --password-stdin {registry_url}")

ECR Authentication Token Retrieved
Registry  : https://797715838180.dkr.ecr.eu-north-1.amazonaws.com
Username  : AWS
Password  : eyJwYXlsb2Fk... [truncated]
Expires   : 2026-07-13 17:08:58.903000+00:00

docker login command:
  aws ecr get-login-password --region eu-north-1 | \
  docker login --username AWS --password-stdin 797715838180.dkr.ecr.eu-north-1.amazonaws.com


## 4.6 Build and Push Docker Image

### Block 14 — AWS-First Build Strategy

**Layer 1 — AWS CodeBuild:** boto3 creates a real build project, uploads source from S3, starts the build in a managed Linux environment with `privilegedMode=True` (Docker-in-Docker), polls until done, and pushes the image to ECR.

**Layer 2 — Local Docker:** if CodeBuild unavailable, falls back to `subprocess` calling `docker build` and `docker push` directly.

**Layer 3 — Print:** always shown so students can run manually in any terminal.

In [14]:
# Block 14 - Build and push Docker images via AWS CodeBuild
# 100% AWS — CodeBuild downloads the source ZIP from S3,
# runs docker build + docker push inside a managed environment.
# No local Docker daemon needed.

BUILD_SUCCEEDED = False

print("Building Docker images via AWS CodeBuild")
print(f"Source : s3://{bucket_name}/{source_zip_key}")
print(f"Target : {ecr_repo_uri}")
print()

try:
    # 1. Create CodeBuild project (safe to re-run — reuses if exists)
    try:
        codebuild.create_project(
            name=cb_project,
            description="Lesson 4 — Build and push Docker images to ECR",
            source={
                "type":     "S3",
                "location": f"{bucket_name}/{source_zip_key}",
            },
            artifacts={"type": "NO_ARTIFACTS"},
            environment={
                "type":           "LINUX_CONTAINER",
                "image":          "aws/codebuild/standard:7.0",
                "computeType":    "BUILD_GENERAL1_SMALL",
                "privilegedMode":  True,
                "environmentVariables": [
                    {"name": "AWS_DEFAULT_REGION", "value": region,        "type": "PLAINTEXT"},
                    {"name": "ECR_REPO_URI",       "value": ecr_repo_uri,  "type": "PLAINTEXT"},
                    {"name": "RUN_ID",             "value": run_id,        "type": "PLAINTEXT"},
                ]
            },
            serviceRole=execution_role_arn,
            logsConfig={
                "cloudWatchLogs": {
                    "status":    "ENABLED",
                    "groupName": f"/lesson4/{project_name}/codebuild",
                }
            },
            tags=[{"key": "Project", "value": project_name}]
        )
        print("CodeBuild project created:", cb_project)
    except codebuild.exceptions.ResourceAlreadyExistsException:
        print("Reusing CodeBuild project:", cb_project)

    # 2. Start the build
    build_resp = codebuild.start_build(projectName=cb_project)
    build_id   = build_resp["build"]["id"]
    print(f"Build started : {build_id}")
    print(f"CloudWatch Logs : /lesson4/{project_name}/codebuild")
    print()

    # 3. Poll until done (every 30s, max 10 min)
    print("Polling build status...")
    cb_status = None
    for attempt in range(20):
        time.sleep(30)
        elapsed   = (attempt + 1) * 30
        builds    = codebuild.batch_get_builds(ids=[build_id])["builds"]
        cb_status = builds[0]["buildStatus"]
        phase     = builds[0].get("currentPhase", "?")
        print(f"  [{elapsed:>3}s] {cb_status:<15} phase={phase}")
        if cb_status in ("SUCCEEDED","FAILED","FAULT","STOPPED","TIMED_OUT"):
            break

    if cb_status == "SUCCEEDED":
        BUILD_SUCCEEDED = True
        print()
        print("Build SUCCEEDED — images pushed to ECR:")
        print(f"  {ecr_repo_uri}:train-{run_id}")
        print(f"  {ecr_repo_uri}:serve-{run_id}")
    else:
        print()
        print(f"Build {cb_status}. Check CloudWatch Logs:")
        print(f"  aws logs tail /lesson4/{project_name}/codebuild --follow")

except Exception as e:
    if "AccessDenied" in str(e) or "AuthorizationError" in str(e):
        print(f"Permission error: {e}")
        print("Fix: ensure codebuild.amazonaws.com is in the IAM role trust policy.")
    else:
        print(f"CodeBuild error: {e}")

print()
print(f"BUILD_SUCCEEDED = {BUILD_SUCCEEDED}")

Building Docker images via AWS CodeBuild
Source : s3://lesson4-containerization-797715838180-eu-north-1/lesson4/container-artifacts/build-source/source.zip
Target : 797715838180.dkr.ecr.eu-north-1.amazonaws.com/lesson4-loan-risk-model

Reusing CodeBuild project: lesson4-docker-build-797715838180
Build started : lesson4-docker-build-797715838180:fb0018dc-0958-4905-86a7-7bacc69cde4f
CloudWatch Logs : /lesson4/lesson4-containerization/codebuild

Polling build status...
  [ 30s] IN_PROGRESS     phase=BUILD
  [ 60s] IN_PROGRESS     phase=BUILD
  [ 90s] IN_PROGRESS     phase=BUILD
  [120s] IN_PROGRESS     phase=BUILD
  [150s] IN_PROGRESS     phase=POST_BUILD
  [180s] SUCCEEDED       phase=COMPLETED

Build SUCCEEDED — images pushed to ECR:
  797715838180.dkr.ecr.eu-north-1.amazonaws.com/lesson4-loan-risk-model:train-20260713-050858
  797715838180.dkr.ecr.eu-north-1.amazonaws.com/lesson4-loan-risk-model:serve-20260713-050858

BUILD_SUCCEEDED = True


In [15]:
logs = boto3.client("logs", region_name="eu-north-1")
group = f"/lesson4/{project_name}/codebuild"

streams = logs.describe_log_streams(
    logGroupName=group,
    orderBy="LastEventTime",
    descending=True,
    limit=1
)["logStreams"]

events = logs.get_log_events(
    logGroupName=group,
    logStreamName=streams[0]["logStreamName"],
    limit=100
)["events"]

for e in events:
    msg = e["message"].rstrip()
    if msg:
        print(msg)

#10 12.11 Collecting mdurl~=0.1
#10 12.12   Downloading mdurl-0.1.2-py3-none-any.whl (10.0 kB)
#10 12.53 Installing collected packages: pytz, websockets, uvloop, urllib3, ujson, tzdata, typing-extensions, tomli, threadpoolctl, six, shellingham, pyyaml, python-multipart, python-dotenv, pygments, prometheus-client, orjson, numpy, mdurl, MarkupSafe, joblib, jmespath, idna, httptools, h11, dnspython, click, certifi, annotated-types, annotated-doc, uvicorn, typing-inspection, scipy, python-dateutil, pydantic-core, markdown-it-py, jinja2, httpcore, exceptiongroup, email_validator, scikit-learn, rich, pydantic, pandas, botocore, anyio, watchfiles, typer, starlette, s3transfer, rich-toolkit, httpx, boto3, fastapi-cli, fastapi
#10 26.69 Successfully installed MarkupSafe-3.0.3 annotated-doc-0.0.4 annotated-types-0.7.0 anyio-4.14.2 boto3-1.34.99 botocore-1.34.162 certifi-2026.6.17 click-8.4.2 dnspython-2.8.0 email_validator-2.3.0 exceptiongroup-1.3.1 fastapi-0.111.0 fastapi-cli-0.0.29 h11-0.16.0 

### Block 15 — Verify Image in ECR

In [16]:
# Block 15 - Verify images in ECR after build

print(f"ECR Repository: {ecr_repo_uri}")
print()
try:
    images = ecr.describe_images(
        repositoryName=ecr_repo_name,
        filter={"tagStatus": "ANY"}
    ).get("imageDetails", [])

    if images:
        print(f"Images in ECR ({len(images)} total):")
        print(f"  {"Tags":<40} {"Size MB":>8}  Pushed")
        print("  " + "-" * 65)
        for img in sorted(images,
                          key=lambda x: x.get("imagePushedAt", dt.datetime.min.replace(tzinfo=None)),
                          reverse=True):
            tags    = ", ".join(img.get("imageTags", ["<untagged>"]))
            size_mb = img.get("imageSizeInBytes", 0) / 1024 / 1024
            pushed  = img.get("imagePushedAt")
            ts      = pushed.strftime("%Y-%m-%d %H:%M") if pushed else "-"
            print(f"  {tags:<40} {size_mb:>8.1f}  {ts}")
    else:
        print("No images yet. Push using Layer 3 commands from Block 14.")

except Exception as e:
    print(f"Could not query ECR: {e}")

ECR Repository: 797715838180.dkr.ecr.eu-north-1.amazonaws.com/lesson4-loan-risk-model

Images in ECR (4 total):
  Tags                                      Size MB  Pushed
  -----------------------------------------------------------------
  serve-20260713-045105                       166.0  2026-07-13 05:11
  train-20260713-045105                       260.1  2026-07-13 05:11
  <untagged>                                  166.0  2026-07-13 05:11
  <untagged>                                  260.1  2026-07-13 05:11


### Block 16 — Register ECS Task Definition

Registering an ECS Task Definition wires the ECR image into real AWS compute. This is how the container image moves from ECR into production serving on ECS Fargate. Works even if the image doesn't exist in ECR yet — ECS validates at launch time.

In [17]:
# Block 16 - Register ECS Task Definition using the ECR image URI

image_uri = f"{ecr_repo_uri}:train-{run_id}"

try:
    resp = ecs.register_task_definition(
        family                   = "lesson4-loan-risk-trainer",
        networkMode              = "awsvpc",
        requiresCompatibilities  = ["FARGATE"],
        cpu                      = "512",
        memory                   = "1024",
        executionRoleArn         = execution_role_arn,
        containerDefinitions=[
            {
                "name":      "loan-risk-trainer",
                "image":     image_uri,
                "cpu":       512,
                "memory":    1024,
                "essential": True,
                "portMappings": [{"containerPort": 8080, "protocol": "tcp"}],
                "environment": [
                    {"name": "MODEL_BUCKET",       "value": bucket_name},
                    {"name": "AWS_DEFAULT_REGION", "value": region},
                    {"name": "RUN_ID",             "value": run_id},
                ],
                "logConfiguration": {
                    "logDriver": "awslogs",
                    "options": {
                        "awslogs-group":        f"/lesson4/{project_name}/trainer",
                        "awslogs-region":        region,
                        "awslogs-stream-prefix": "ecs"
                    }
                },
                "healthCheck": {
                    "command":     ["CMD-SHELL","curl -f http://localhost:8080/health || exit 1"],
                    "interval":    30,
                    "timeout":     10,
                    "retries":     3,
                    "startPeriod": 15,
                }
            }
        ],
        tags=[{"key": "Project", "value": project_name}, {"key": "RunId", "value": run_id}]
    )

    td      = resp["taskDefinition"]
    td_arn  = td["taskDefinitionArn"]

    print("ECS Task Definition registered")
    print()
    print("ARN      :", td_arn)
    print("Family   :", td["family"])
    print("Revision :", td["revision"])
    print("Status   :", td["status"])
    print("CPU      :", td["cpu"])
    print("Memory   :", td["memory"])
    print("Image    :", image_uri)
    print()
    print("To run on Fargate:")
    print("  aws ecs run-task \\")
    print("    --cluster <your-cluster> \\")
    print("    --task-definition lesson4-loan-risk-trainer \\")
    print("    --launch-type FARGATE \\")
    print("    --network-configuration '{\"awsvpcConfiguration\":{\"subnets\":[\"subnet-xxx\"],\"assignPublicIp\":\"ENABLED\"}}'") 

except Exception as e:
    if "AccessDenied" in str(e) or "AuthorizationError" in str(e):
        print(f"ECS Task Definition skipped (permission denied): {type(e).__name__}")
        print("  Add ecs:RegisterTaskDefinition to the execution role.")
    else:
        print(f"ECS Task Definition error: {e}")

ECS Task Definition skipped (permission denied): AccessDeniedException
  Add ecs:RegisterTaskDefinition to the execution role.


## 4.7 CloudWatch Observability

### Block 17 — CloudWatch Log Groups

In [18]:
# Block 17 - Create CloudWatch log groups for all container services

log_groups = {
    f"/lesson4/{project_name}/trainer":   7,
    f"/lesson4/{project_name}/server":   14,
    f"/lesson4/{project_name}/gpu":       7,
    f"/lesson4/{project_name}/codebuild": 7,
}

for group_name, retention_days in log_groups.items():
    try:
        logs_client.create_log_group(logGroupName=group_name)
        print(f"  Created : {group_name}")
    except logs_client.exceptions.ResourceAlreadyExistsException:
        print(f"  Reusing : {group_name}")
    logs_client.put_retention_policy(
        logGroupName=group_name, retentionInDays=retention_days
    )
    print(f"    Retention: {retention_days} days")

print("\nCloudWatch log groups ready")

  Reusing : /lesson4/lesson4-containerization/trainer
    Retention: 7 days
  Reusing : /lesson4/lesson4-containerization/server
    Retention: 14 days
  Reusing : /lesson4/lesson4-containerization/gpu
    Retention: 7 days
  Reusing : /lesson4/lesson4-containerization/codebuild
    Retention: 7 days

CloudWatch log groups ready


### Block 18 — Publish Metrics to CloudWatch

In [19]:
# Block 18 - Publish container registry and build metrics

metric_data = [
    {"MetricName": "DockerfileCount",    "Value": float(len(artifacts))},
    {"MetricName": "ECRRepositoryReady", "Value": 1.0},
    {"MetricName": "BuildAttempted",     "Value": 1.0},
    {"MetricName": "BuildSucceeded",     "Value": 1.0 if BUILD_SUCCEEDED else 0.0},
]

cloudwatch.put_metric_data(
    Namespace=namespace,
    MetricData=[
        {**m, "Unit": "None", "Dimensions": [{"Name": "Project", "Value": project_name}]}
        for m in metric_data
    ]
)
print(f"CloudWatch metrics published — namespace: {namespace}")
for m in metric_data:
    icon = "✅" if m["Value"] >= 1.0 else "📋"
    print(f"  {icon}  {m['MetricName']:<28}: {m['Value']}")

CloudWatch metrics published — namespace: Lesson4/Containerization
  ✅  DockerfileCount             : 6.0
  ✅  ECRRepositoryReady          : 1.0
  ✅  BuildAttempted              : 1.0
  ✅  BuildSucceeded              : 1.0


## 4.8 Summary

### Block 19 — Image Manifest and S3 Artifact Listing

In [20]:
# Block 19 - Generate manifest and list all S3 artifacts

manifest = {
    "project":     project_name,
    "run_id":      run_id,
    "aws_region":  region,
    "aws_account": account_id,
    "build": {
        "method":     "AWS CodeBuild",
        "succeeded":  BUILD_SUCCEEDED,
        "cb_project": cb_project,
    },
    "ecr": {
        "repository_name": ecr_repo_name,
        "repository_uri":  ecr_repo_uri,
        "image_tags": [f"train-{run_id}", f"serve-{run_id}"],
    },
    "ecs": {"task_family": "lesson4-loan-risk-trainer"},
    "s3_artifacts": {
        "train_dockerfile": f"s3://{bucket_name}/{dockerfile_key}",
        "serve_dockerfile": f"s3://{bucket_name}/{multistage_key}",
        "gpu_dockerfile":   f"s3://{bucket_name}/{gpu_key}",
        "build_zip":        f"s3://{bucket_name}/{source_zip_key}",
    },
    "cloudwatch": {"namespace": namespace, "log_groups": list(log_groups.keys())},
    "created_at_utc": dt.datetime.utcnow().isoformat()
}
manifest_path = local_dir / f"image_manifest_{run_id}.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
s3.upload_file(str(manifest_path), bucket_name, manifest_key)
print("Manifest uploaded:", f"s3://{bucket_name}/{manifest_key}")
print()
print(json.dumps(manifest, indent=2))
print()

# List all S3 artifacts
print("S3 Artefacts:")
objects = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix).get("Contents", [])
for obj in objects:
    size_kb = obj["Size"] / 1024
    ts      = obj["LastModified"].strftime("%Y-%m-%d %H:%M")
    print(f"  {obj['Key']}")
    print(f"    {size_kb:.1f} KB  |  {ts} UTC")
print(f"\nTotal artefacts: {len(objects)}")

Manifest uploaded: s3://lesson4-containerization-797715838180-eu-north-1/lesson4/container-artifacts/manifest/image_manifest_20260713-050858.json

{
  "project": "lesson4-containerization",
  "run_id": "20260713-050858",
  "aws_region": "eu-north-1",
  "aws_account": "797715838180",
  "build": {
    "method": "AWS CodeBuild",
    "succeeded": true,
    "cb_project": "lesson4-docker-build-797715838180"
  },
  "ecr": {
    "repository_name": "lesson4-loan-risk-model",
    "repository_uri": "797715838180.dkr.ecr.eu-north-1.amazonaws.com/lesson4-loan-risk-model",
    "image_tags": [
      "train-20260713-050858",
      "serve-20260713-050858"
    ]
  },
  "ecs": {
    "task_family": "lesson4-loan-risk-trainer"
  },
  "s3_artifacts": {
    "train_dockerfile": "s3://lesson4-containerization-797715838180-eu-north-1/lesson4/container-artifacts/dockerfiles/Dockerfile.train",
    "serve_dockerfile": "s3://lesson4-containerization-797715838180-eu-north-1/lesson4/container-artifacts/dockerfiles/Do

## 4.9 Cleanup

### Block 20 — Optional Cleanup

In [21]:
# Block 20 - Optional cleanup

CLEANUP = False

if CLEANUP:
    # S3
    objs = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix).get("Contents", [])
    if objs:
        s3.delete_objects(Bucket=bucket_name,
            Delete={"Objects": [{"Key": o["Key"]} for o in objs]})
        print(f"Deleted {len(objs)} S3 objects")

    # CloudWatch log groups
    for group_name in log_groups:
        try:
            logs_client.delete_log_group(logGroupName=group_name)
            print(f"Deleted: {group_name}")
        except Exception: pass

    # ECR
    ecr.delete_repository(repositoryName=ecr_repo_name, force=True)
    print(f"Deleted ECR: {ecr_repo_name}")

    # CodeBuild
    try:
        codebuild.delete_project(name=cb_project)
        print(f"Deleted CodeBuild: {cb_project}")
    except Exception: pass

    # ECS task definitions
    try:
        tds = ecs.list_task_definitions(familyPrefix="lesson4-loan-risk-trainer").get("taskDefinitionArns",[])
        for td_arn in tds:
            ecs.deregister_task_definition(taskDefinition=td_arn)
        print(f"Deregistered {len(tds)} ECS task def(s)")
    except Exception: pass

else:
    print("Cleanup skipped — set CLEANUP = True to delete all resources")

Cleanup skipped — set CLEANUP = True to delete all resources


### Final Checklist — Real AWS Services Used

| Service | Operations | Block |
|---|---|---|
| **AWS S3** | create_bucket, upload_file, upload_fileobj (ZIP), list_objects_v2 | 4, 10, 19 |
| **Amazon ECR** | create_repository, put_image_scanning_configuration, put_lifecycle_policy, get_authorization_token, describe_images | 11–13, 15 |
| **AWS CodeBuild** | create_project, start_build, batch_get_builds (polls real build) | 14 Layer 1 |
| **Amazon ECS** | register_task_definition (Fargate + awslogs) | 16 |
| **CloudWatch Logs** | create_log_group, put_retention_policy | 17 |
| **CloudWatch Metrics** | put_metric_data | 18 |
| **AWS STS** | get_caller_identity, derive execution role ARN for CodeBuild | 2 |

**Build result:** Check `BUILD_SUCCEEDED` and `BUILD_METHOD` after Block 14.
